In [ ]:
# load data for CSI

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

excel_path = Path("/Users/julesschleinitz/Desktop/Code/BNNY/fluoride_modeling_properties_w_experimental_data.xlsx")

cs1_bncl_ddg = pd.read_excel(excel_path, sheet_name="CS2_BnCl_ddg")
cs1_az_ddg = pd.read_excel(excel_path, sheet_name="CS2_Az_ddg")

print("Loaded CS1_BnCl_ddg shape:", cs1_bncl_ddg.shape)
print("Loaded CS1_Az_ddg shape:", cs1_az_ddg.shape)


## preprocess the dataframes to keep only relevant columns and add 'type' column to cs1_az_ddg based on cs1_bncl_ddg
cs1_bncl_ddg.drop(columns=[0, 1, 2, 4, 5, 6, 7, 8, 10, 11, 12, 13], inplace=True)
# use first data row as header, then remove it from the data
cs1_bncl_ddg.columns = cs1_bncl_ddg.iloc[0]
cs1_bncl_ddg = cs1_bncl_ddg.iloc[1:].reset_index(drop=True)
cs1_bncl_ddg["rxn"] = "BnCl"

cs1_az_ddg.drop(columns=[0, 1, 2, 4, 5, 6, 7, 9, 10, 11, 12], inplace=True)
cs1_az_ddg.columns = cs1_az_ddg.iloc[0]
cs1_az_ddg.drop(columns=["experimental_smiles"], inplace=True)
cs1_az_ddg = cs1_az_ddg.iloc[1:].reset_index(drop=True)
cs1_az_ddg["rxn"] = "Az"

print("cs1_bncl_ddg columns after preprocessing:", cs1_bncl_ddg.columns[:20])
print("cs1_az_ddg columns after preprocessing:", cs1_az_ddg.columns[:20])

## concatenate the dataframes from the two Csp3 coupling partners
cs1 = pd.concat([cs1_bncl_ddg, cs1_az_ddg], ignore_index=True)
cs1["type"] = "N"

## add a preprocessing step to assign all missing types to the ligands classes
from rdkit import Chem
from rdkit.Chem import Draw

pyox_sub = Chem.MolFromSmarts("C1(=NCCO1)c2ncccc2")
biox_sub = Chem.MolFromSmarts("C1(=NCCO1)C2=NCCO2")
box__sub = Chem.MolFromSmarts("C1(=NCCO1)CC2=NCCO2")
biim_sub = Chem.MolFromSmarts("C1(=NCCN1)C2=NCCN2")
quin_sub = Chem.MolFromSmarts("C1(c(ccc2)c3c2cccn3)=NCCN1")
smiles_n = []
for i, row in cs1.iterrows():
    if row["type"] == "N":
        #print(f"Row {i} has type 'N' and ligandID {row['ligandID']}")
        lig_mol = Chem.MolFromSmiles(row["computational_smiles"])
        if lig_mol is None:
            print(f"Could not parse SMILES for row {i}: {row['computational_smiles']}")
        elif lig_mol.HasSubstructMatch(pyox_sub):
            #print(f"Row {i} matches pyox substructure")
            cs1.at[i, "type"] = "pyox"
        elif lig_mol.HasSubstructMatch(biox_sub):
            #print(f"Row {i} matches biox substructure")
            cs1.at[i, "type"] = "biox"
        elif lig_mol.HasSubstructMatch(box__sub):
            #print(f"Row {i} matches box substructure")
            cs1.at[i, "type"] = "box"
        elif lig_mol.HasSubstructMatch(biim_sub):
            #print(f"Row {i} matches biim substructure")
            cs1.at[i, "type"] = "biim"
        elif lig_mol.HasSubstructMatch(quin_sub):
            #print(f"Row {i} matches quin substructure")
            cs1.at[i, "type"] = "quin"
            print(f"Row {i} matches quin substructure, setting type to 'quin'")
            print(f"SMILES: {row['computational_smiles']}")
        else:
            print(f"Row {i} does not match any substructure, keeping type 'N'")
            print(f"SMILES: {row['computational_smiles']}")

        smiles_n.append(row["computational_smiles"])


# Draw.MolsToGridImage([Chem.MolFromSmiles(smiles) for smiles in smiles_n], molsPerRow=5)


In [ ]:
cs1.head()

In [ ]:
dict_smiles_to_substituent = {'c1ccc(C[C@H]2COC(C3=N[C@@H](Cc4ccccc4)CO3)=N2)cc1': 'Bn',
 'CC(C)[C@H]1COC(C2=N[C@@H](C(C)C)CO2)=N1': 'iPr',
 'C1CCC([C@H]2COC(C3=N[C@@H](C4CCCCC4)CO3)=N2)CC1': 'Cy',
 'c1ccc([C@H]2COC(C3=N[C@@H](c4ccccc4)CO3)=N2)cc1': 'Ph',
 'CCCC(CCC)[C@H]1COC(C2=N[C@@H](C(CCC)CCC)CO2)=N1': 'iHept',
 'C[C@H]1COC(C2=N[C@@H](C)CO2)=N1': 'Me',
 'CC[C@H](C)[C@H]1COC(C2=N[C@@H]([C@@H](C)CC)CO2)=N1': 'secBu',
 'CC(C)[C@H]1CN(c2cccc(C(C)(C)C)c2)C(C2=N[C@@H](C(C)C)CN2c2cccc(C(C)(C)C)c2)=N1': 'iPr',
 'c1ccc([C@H]2COC(c3ccc4ccccc4n3)=N2)cc1': 'Ph',
 'CC(C)(C)[C@H]1COC(c2ccc(C(F)(F)F)cn2)=N1': 'tBu',
 'FC(F)(F)c1ccc(C2=N[C@@H](c3ccccc3)[C@@H](c3ccccc3)O2)nc1': 'Ph',
 'Cc1cccc(C2=N[C@@H](C(C)C)CO2)n1': 'iPr',
 'CC(C)[C@H]1COC(c2ccc3ccccc3n2)=N1': 'iPr',
 'c1ccc(C[C@H]2COC(C3=N[C@@H](Cc4ccccc4)CO3)=N2)cc1': 'Bn',
 'CC(C)[C@H]1CN(C2CCCCC2)C(C2=N[C@@H](C(C)C)CN2C2CCCCC2)=N1': 'iPr',
 'c1ccc([C@H]2COC(c3ccccn3)=N2)cc1': 'Ph',
 'CC(C)[C@H]1CN2CCN3C[C@H](C(C)C)N=C3C2=N1': 'iPr',
 'CC(C)[C@H]1COC(c2ccccn2)=N1': 'iPr',
 'CC(C)(C)[C@H]1CN(c2cc(C(F)(F)F)cc(C(F)(F)F)c2)C(c2cccc3cccnc23)=N1': 'tBu',
 'c1ccc([C@H]2COC(C3(C4=N[C@@H](c5ccccc5)CO4)CCCC3)=N2)cc1': 'Ph',
 'CC(C)(C1=N[C@@H](c2ccccc2)CO1)C1=N[C@@H](c2ccccc2)CO1': 'Ph',
 'CC(C)(C1=N[C@@H](Cc2ccccc2)CO1)C1=N[C@@H](Cc2ccccc2)CO1': 'Bn',
 'CC(C)(C1=N[C@@H](C(C)(C)C)CO1)C1=N[C@@H](C(C)(C)C)CO1': 'tBu',
 'CC(C)C[C@H]1COC(C(C)(C)C2=N[C@@H](CC(C)C)CO2)=N1': 'iBu',
 'c1ccc(C[C@H]2COC(C3(C4=N[C@@H](Cc5ccccc5)CO4)CC3)=N2)cc1': 'Bn',
 'CC(C)(C)c1ccc(CC(Cc2ccc(C(C)(C)C)cc2)(C2=N[C@@H](c3ccccc3)CO2)C2=N[C@@H](c3ccccc3)CO2)cc1': 'Ph',
 'c1ccc2c(c1)C[C@H]1OC(C3(C4=N[C@H]5c6ccccc6C[C@H]5O4)CC3)=N[C@@H]21': 'Inda',
 'CC(C)(C)[C@H]1COC(C2(C3=N[C@@H](C(C)(C)C)CO3)CCC2)=N1': 'tBu',
 'CC(C)[C@H]1COC(C(C)(C)C2=N[C@@H](C(C)C)CO2)=N1': 'iPr',
}

for key, value in dict_smiles_to_substituent.items():
    cs1.loc[cs1["computational_smiles"] == key, "substituent"] = value

for i, row in cs1.iterrows():
    if pd.isna(row["substituent"]):
        print(f"Row {i} has no substituent assigned and SMILES {row['computational_smiles']}")

Draw.MolsToGridImage([Chem.MolFromSmiles(smiles) for smiles in cs1["computational_smiles"]], molsPerRow=5, legends=cs1["substituent"].tolist())

In [ ]:
for s in cs1["computational_smiles"]:
    try:
        m = Chem.MolFromSmiles(s)
        if m is None:
            print(f"Could not parse SMILES: {s} - None")
    except:
        print(f"Could not parse SMILES: {s}")

## CS2 linear modeling pipeline (DFT, Morgan, type OHE)
This section builds descriptor-family-specific linear models for `ddg` with:
- Descriptor families: DFT slice, Morgan fingerprint, type one-hot
- Outer validation regimes: random split, y-equidistant split, leave-one-ligand-out, leave-one-type-out, leave-one-substituent-out
- Inner model selection: y-equidistant inner splits with max 4 selected descriptors and overfitting-aware score

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

from sklearn.base import clone
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# -----------------------------
# 0) Selection configuration
# -----------------------------
# Options: "r2", "mae", "balanced"
selection_mode          = "balanced"
inner_cv_n_splits       = 5
spec_pool_size          = 150000
max_selected_features   = 4
# Global DFT preprocessing: remove one feature from highly correlated pairs before any model fitting.
dft_correlation_threshold = 0.7


def remove_correlated_features(df, threshold):
    if threshold is None:
        return df.copy(), []

    corr = df.corr().abs()
    upper_triangle = corr.mask(np.tril(np.ones(corr.shape, dtype=bool)))
    keep_mask = ~(upper_triangle > threshold).any()
    keep_cols = keep_mask[keep_mask].index.tolist()
    removed_cols = [col for col in df.columns if col not in keep_cols]
    return df.loc[:, keep_cols].copy(), removed_cols


def canonical_descriptor_key(model_name, selected_names):
    return f"{model_name}__{'|'.join(sorted(map(str, selected_names.tolist())))}"


# -----------------------------
# 1) Build target + feature families
# -----------------------------
required_cols = ["ddg_flipped", "computational_smiles", "ligandID", "type", "rxn", "HOMO_Boltz", "Visible_volume_Ni(Å³)_low_E"]
missing_cols = [c for c in required_cols if c not in cs1.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in cs1: {missing_cols}")

work = cs1.copy()
work = work.dropna(subset=["ddg_flipped", "computational_smiles", "ligandID", "type"]).reset_index(drop=True)
work["ddg_flipped"] = pd.to_numeric(work["ddg_flipped"], errors="coerce")
work = work.dropna(subset=["ddg_flipped"]).reset_index(drop=True)
if "substituent" not in work.columns:
    work["substituent"] = "unknown"
work["substituent"] = work["substituent"].fillna("unknown").astype(str)
work.loc[work["substituent"].str.strip() == "", "substituent"] = "unknown"

# DFT descriptor family
X_dft = work.loc[:, "HOMO_Boltz":"Visible_volume_Ni(Å³)_low_E"].apply(pd.to_numeric, errors="coerce")
X_dft = X_dft.fillna(X_dft.median(numeric_only=True)).fillna(0.0)
dft_feature_count_before = X_dft.shape[1]
X_dft, dft_removed_features = remove_correlated_features(X_dft, dft_correlation_threshold)
dft_feature_count_after = X_dft.shape[1]

# Morgan fingerprint family
morgan_bits = 2048
morgan_radius = 2
morgan_generator = GetMorganGenerator(radius=morgan_radius, fpSize=morgan_bits)
fp_rows = []
valid_idx = []
for idx, smi in enumerate(work["computational_smiles"].tolist()):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    bv = morgan_generator.GetFingerprint(mol)
    arr = np.zeros((morgan_bits,), dtype=int)
    DataStructs.ConvertToNumpyArray(bv, arr)
    fp_rows.append(arr)
    valid_idx.append(idx)

if len(valid_idx) != len(work):
    work = work.iloc[valid_idx].reset_index(drop=True)
    X_dft = X_dft.iloc[valid_idx].reset_index(drop=True)

X_morgan = pd.DataFrame(fp_rows, columns=[f"morgan_{i}" for i in range(morgan_bits)])
X_type = pd.get_dummies(work["type"].astype(str), prefix="type", dtype=int)

y = work["ddg_flipped"].to_numpy()

feature_sets = {
    "DFT": X_dft.reset_index(drop=True),
    "Morgan": X_morgan.reset_index(drop=True),
    "TypeOHE": X_type.reset_index(drop=True),
}

meta = work[["ligandID", "type", "rxn", "substituent"]].reset_index(drop=True)


# -----------------------------
# 2) Split generators
# -----------------------------
def generate_random_splits_by_group(groups, test_size=0.2, n_repeats=5, seed=42):
    splits = []
    group_arr = np.asarray(groups)
    unique_groups = pd.Series(group_arr).dropna().unique()

    if len(unique_groups) < 2:
        return splits

    all_idx = np.arange(len(group_arr))
    test_group_count = max(1, int(np.ceil(test_size * len(unique_groups))))
    if test_group_count >= len(unique_groups):
        test_group_count = len(unique_groups) - 1

    for rep in range(n_repeats):
        grp_tr, grp_te = train_test_split(
            unique_groups,
            test_size=test_group_count,
            random_state=seed + rep,
            shuffle=True,
        )
        te_mask = np.isin(group_arr, grp_te)
        te = all_idx[te_mask]
        tr = all_idx[~te_mask]
        if len(te) == 0 or len(tr) == 0:
            continue
        splits.append((tr, te, f"random_{rep}"))
    return splits


def generate_leave_one_group_splits(groups):
    splits = []
    all_idx = np.arange(len(groups))
    for g in pd.Series(groups).dropna().unique():
        te = np.where(np.array(groups) == g)[0]
        tr = np.setdiff1d(all_idx, te)
        if len(te) == 0 or len(tr) == 0:
            continue
        splits.append((tr, te, str(g)))
    return splits


def generate_y_equidistant_splits_by_group(y_values, groups, n_splits=5):
    y_arr = np.asarray(y_values, dtype=float)
    group_arr = np.asarray(groups)

    grp_df = pd.DataFrame({"group": group_arr, "y": y_arr}).dropna(subset=["group", "y"])
    if grp_df.empty:
        return []

    grp_mean_y = grp_df.groupby("group", as_index=False)["y"].mean()
    grp_sorted = grp_mean_y.sort_values("y")["group"].to_numpy()

    effective_splits = min(n_splits, len(grp_sorted))
    if effective_splits < 2:
        return []

    all_idx = np.arange(len(group_arr))
    splits = []
    for fold in range(effective_splits):
        te_groups = grp_sorted[fold::effective_splits]
        te_mask = np.isin(group_arr, te_groups)
        te = all_idx[te_mask]
        tr = all_idx[~te_mask]
        if len(te) == 0 or len(tr) == 0:
            continue
        splits.append((tr, te, f"y_equidistant_{fold}"))
    return splits


def generate_y_equidistant_index_splits(y_values, n_splits=5):
    y_arr = np.asarray(y_values, dtype=float)
    n_samples = len(y_arr)
    effective_splits = min(n_splits, n_samples)
    if effective_splits < 2:
        return []

    all_idx = np.arange(n_samples)
    sort_idx = np.argsort(y_arr)
    splits = []
    for fold in range(effective_splits):
        val_idx = sort_idx[fold::effective_splits]
        tr_idx = np.setdiff1d(all_idx, val_idx)
        if len(val_idx) == 0 or len(tr_idx) == 0:
            continue
        splits.append((tr_idx, val_idx))
    return splits


outer_splits = {
    "random": generate_random_splits_by_group(meta["ligandID"].tolist(), test_size=0.2, n_repeats=5, seed=42),
    "y_equidistant": generate_y_equidistant_splits_by_group(y, meta["ligandID"].tolist(), n_splits=5),
    "leave_one_ligandID_out": generate_leave_one_group_splits(meta["ligandID"].tolist()),
    "leave_one_type_out": generate_leave_one_group_splits(meta["type"].tolist()),
    "leave_one_substituent_out": generate_leave_one_group_splits(meta["substituent"].tolist()),
}


# -----------------------------
# 3) Inner selection utilities
# -----------------------------
def safe_r2(y_true, y_pred):
    if len(y_true) < 2:
        return np.nan
    if np.isclose(np.std(y_true), 0.0):
        return np.nan
    return r2_score(y_true, y_pred)


def stable_group_r2(y_true, raw_r2, split_name):
    # For leave-one-group settings, suppress fragile R2 from tiny held-out sets.
    if split_name in {"leave_one_ligandID_out", "leave_one_type_out", "leave_one_substituent_out"} and len(y_true) < 3:
        return np.nan
    if pd.isna(raw_r2) or np.isinf(raw_r2):
        return np.nan
    return raw_r2


def rank_candidate(train_r2, train_mae, val_q2, val_mae, mode):
    # Higher is better in all modes.
    if mode == "r2":
        return -1.0 if np.isnan(val_q2) else val_q2
    if mode == "mae":
        return -val_mae
    if mode == "balanced":
        v_q2 = -1.0 if np.isnan(val_q2) else val_q2
        t_r2 = -1.0 if np.isnan(train_r2) else train_r2
        return (
            v_q2
            - val_mae
            - 0.5 * abs(t_r2 - v_q2)
            - 0.5 * abs(train_mae - val_mae)
        )
    raise ValueError("selection_mode must be one of: 'r2', 'mae', 'balanced'")


def fit_evaluate_linear_model(X_train, y_train, X_eval, model, selected_k):
    selector = SelectKBest(score_func=f_regression, k=selected_k)
    X_train_sel = selector.fit_transform(X_train, y_train)
    X_eval_sel = selector.transform(X_eval)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_eval_scaled = scaler.transform(X_eval_sel)

    fit_model = clone(model)
    fit_model.fit(X_train_scaled, y_train)
    pred_train = fit_model.predict(X_train_scaled)
    pred_eval = fit_model.predict(X_eval_scaled)

    return selector, scaler, fit_model, pred_train, pred_eval


def evaluate_candidate_cv(X_outer_train, y_outer_train, model, k, mode):
    inner_splits = generate_y_equidistant_index_splits(y_outer_train, n_splits=inner_cv_n_splits)
    if len(inner_splits) < 2:
        return None

    tr_r2_list, tr_mae_list = [], []
    val_y_all, val_pred_all = [], []

    for inner_tr_idx, inner_val_idx in inner_splits:
        X_inner_train, X_inner_val = X_outer_train[inner_tr_idx], X_outer_train[inner_val_idx]
        y_inner_train, y_inner_val = y_outer_train[inner_tr_idx], y_outer_train[inner_val_idx]

        try:
            _, _, _, pred_tr, pred_val = fit_evaluate_linear_model(
                X_inner_train, y_inner_train, X_inner_val, model, k
            )
        except Exception:
            continue

        tr_r2_list.append(safe_r2(y_inner_train, pred_tr))
        tr_mae_list.append(mean_absolute_error(y_inner_train, pred_tr))
        val_y_all.extend(y_inner_val.tolist())
        val_pred_all.extend(pred_val.tolist())

    if len(val_pred_all) == 0:
        return None

    train_r2_mean = float(np.nanmean(tr_r2_list))
    train_mae_mean = float(np.mean(tr_mae_list))

    val_y_all = np.array(val_y_all, dtype=float)
    val_pred_all = np.array(val_pred_all, dtype=float)
    val_q2_pooled = safe_r2(val_y_all, val_pred_all)
    val_mae_pooled = mean_absolute_error(val_y_all, val_pred_all)

    rank_score = rank_candidate(train_r2_mean, train_mae_mean, val_q2_pooled, val_mae_pooled, mode=mode)

    return {
        "selection_mode": mode,
        "score": rank_score,
        "k": k,
        "train_r2": train_r2_mean,
        "train_mae": train_mae_mean,
        "val_q2": val_q2_pooled,
        "val_mae": val_mae_pooled,
        "inner_cv_evals": len(val_pred_all),
        "inner_cv_splits": len(inner_splits),
        "inner_cv_repeats": 1,
    }


def get_model_candidates():
    candidates = [("LinearRegression", LinearRegression())]
    for a in [0.01, 0.1, 1.0, 10.0]:
        candidates.append((f"Ridge_alpha_{a}", Ridge(alpha=a, random_state=42)))
    for a in [0.001, 0.01, 0.1]:
        candidates.append((f"Lasso_alpha_{a}", Lasso(alpha=a, random_state=42, max_iter=20000)))
    for a in [0.1, 1.0, 10.0]:
        candidates.append((f"KernelRidge_alpha_{a}", KernelRidge(alpha=a, kernel="rbf")))
    return candidates


def build_candidate_result(X_outer_train, y_outer_train, X_outer_test, y_outer_test, feature_names, model_name, model, k, mode):
    cv_eval = evaluate_candidate_cv(
        X_outer_train=X_outer_train,
        y_outer_train=y_outer_train,
        model=model,
        k=k,
        mode=mode,
    )
    if cv_eval is None:
        return None

    selector = SelectKBest(score_func=f_regression, k=k)
    X_tr_sel = selector.fit_transform(X_outer_train, y_outer_train)
    X_te_sel = selector.transform(X_outer_test)
    selected_names = feature_names[selector.get_support()]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr_sel)
    X_te_scaled = scaler.transform(X_te_sel)

    final_model = clone(model)
    final_model.fit(X_tr_scaled, y_outer_train)
    pred_outer_train = final_model.predict(X_tr_scaled)
    pred_outer_test = final_model.predict(X_te_scaled)

    coef = getattr(final_model, "coef_", None)
    coef_map = {}
    if coef is not None:
        flat_coef = np.ravel(coef)
        coef_map = {n: float(c) for n, c in zip(selected_names, flat_coef)}

    outer_test_r2 = safe_r2(y_outer_test, pred_outer_test)

    return {
        **cv_eval,
        "model_name": model_name,
        "model": model,
        "k_selected": int(k),
        "selected_descriptors": list(selected_names),
        "spec_key": canonical_descriptor_key(model_name, selected_names),
        "coefficients": coef_map,
        "intercept": float(getattr(final_model, "intercept_", np.nan)),
        "outer_train_r2": safe_r2(y_outer_train, pred_outer_train),
        "outer_train_mae": mean_absolute_error(y_outer_train, pred_outer_train),
        "outer_test_r2": outer_test_r2,
        "outer_test_mae": mean_absolute_error(y_outer_test, pred_outer_test),
        "pred_outer_test": pred_outer_test,
    }


# -----------------------------
# 4) Outer-loop run + reporting
# -----------------------------
records = []
prediction_records = []
fixed_spec_records = []
fixed_spec_prediction_records = []
validation_labels_by_regime = {
    regime_name: sorted({split[2] for split in split_list})
    for regime_name, split_list in outer_splits.items()
}
leave_one_type_labels = validation_labels_by_regime.get("leave_one_type_out", [])

for descriptor_name, X_df in feature_sets.items():
    X_np = X_df.to_numpy(dtype=float)
    feature_names = np.array(X_df.columns.tolist())

    for split_name, split_list in outer_splits.items():
        for tr_idx, te_idx, held_out_label in split_list:
            X_outer_train, X_outer_test = X_np[tr_idx], X_np[te_idx]
            y_outer_train, y_outer_test = y[tr_idx], y[te_idx]

            if len(y_outer_train) < 5:
                continue

            max_k = min(max_selected_features, X_df.shape[1])
            if max_k < 1:
                continue

            candidate_pool = []
            for k in range(1, max_k + 1):
                for model_name, model in get_model_candidates():
                    candidate_result = build_candidate_result(
                        X_outer_train=X_outer_train,
                        y_outer_train=y_outer_train,
                        X_outer_test=X_outer_test,
                        y_outer_test=y_outer_test,
                        feature_names=feature_names,
                        model_name=model_name,
                        model=model,
                        k=k,
                        mode=selection_mode,
                    )
                    if candidate_result is not None:
                        candidate_pool.append(candidate_result)

            if not candidate_pool:
                continue

            candidate_pool = sorted(candidate_pool, key=lambda item: item["score"], reverse=True)
            best = candidate_pool[0]

            for sample_index, y_true_i, y_pred_i in zip(te_idx.tolist(), y_outer_test.tolist(), best["pred_outer_test"].tolist()):
                prediction_records.append({
                    "selection_mode": best["selection_mode"],
                    "descriptor_family": descriptor_name,
                    "validation_regime": split_name,
                    "held_out_label": held_out_label,
                    "sample_index": int(sample_index),
                    "y_true": float(y_true_i),
                    "y_pred": float(y_pred_i),
                })

            rec = {
                "selection_mode": best["selection_mode"],
                "inner_cv_splits": best["inner_cv_splits"],
                "inner_cv_repeats": best["inner_cv_repeats"],
                "inner_cv_evals": best["inner_cv_evals"],
                "descriptor_family": descriptor_name,
                "validation_regime": split_name,
                "held_out_label": held_out_label,
                "model": best["model_name"],
                "k_selected": best["k_selected"],
                "selected_descriptors": best["selected_descriptors"],
                "coefficients": best["coefficients"],
                "intercept": best["intercept"],
                "inner_cv_train_r2_mean": best["train_r2"],
                "inner_cv_train_mae_mean": best["train_mae"],
                "inner_cv_val_q2": best["val_q2"],
                "inner_cv_val_mae": best["val_mae"],
                "outer_train_r2": best["outer_train_r2"],
                "outer_train_mae": best["outer_train_mae"],
                "outer_test_r2": best["outer_test_r2"],
                "outer_test_r2_stable": stable_group_r2(y_outer_test, best["outer_test_r2"], split_name),
                "outer_test_mae": best["outer_test_mae"],
                "n_outer_train": int(len(y_outer_train)),
                "n_outer_test": int(len(y_outer_test)),
            }
            records.append(rec)

            top_candidates = candidate_pool[: min(spec_pool_size, len(candidate_pool))]
            for rank_in_fold, candidate in enumerate(top_candidates, start=1):
                fixed_spec_records.append({
                    "selection_mode": candidate["selection_mode"],
                    "descriptor_family": descriptor_name,
                    "validation_regime": split_name,
                    "held_out_label": held_out_label,
                    "rank_in_fold": rank_in_fold,
                    "spec_key": candidate["spec_key"],
                    "model": candidate["model_name"],
                    "k_selected": candidate["k_selected"],
                    "selected_descriptors": candidate["selected_descriptors"],
                    "inner_cv_train_r2_mean": candidate["train_r2"],
                    "inner_cv_train_mae_mean": candidate["train_mae"],
                    "inner_cv_val_q2": candidate["val_q2"],
                    "inner_cv_val_mae": candidate["val_mae"],
                    "inner_cv_splits": candidate["inner_cv_splits"],
                    "inner_cv_repeats": candidate["inner_cv_repeats"],
                    "inner_cv_evals": candidate["inner_cv_evals"],
                })

                for sample_index, y_true_i, y_pred_i in zip(te_idx.tolist(), y_outer_test.tolist(), candidate["pred_outer_test"].tolist()):
                    fixed_spec_prediction_records.append({
                        "selection_mode": candidate["selection_mode"],
                        "descriptor_family": descriptor_name,
                        "validation_regime": split_name,
                        "held_out_label": held_out_label,
                        "rank_in_fold": rank_in_fold,
                        "spec_key": candidate["spec_key"],
                        "model": candidate["model_name"],
                        "k_selected": candidate["k_selected"],
                        "selected_descriptors": candidate["selected_descriptors"],
                        "sample_index": int(sample_index),
                        "y_true": float(y_true_i),
                        "y_pred": float(y_pred_i),
                    })

results_df = pd.DataFrame(records)
if results_df.empty:
    raise RuntimeError("No models were produced. Check data and split settings.")

pred_df = pd.DataFrame(prediction_records)
if pred_df.empty:
    raise RuntimeError("No outer predictions were produced. Check model evaluation loop.")

fixed_spec_df = pd.DataFrame(fixed_spec_records)
fixed_spec_pred_df = pd.DataFrame(fixed_spec_prediction_records)
leave_one_type_spec_df = fixed_spec_df[fixed_spec_df["validation_regime"] == "leave_one_type_out"].copy()
leave_one_type_spec_pred_df = fixed_spec_pred_df[fixed_spec_pred_df["validation_regime"] == "leave_one_type_out"].copy()


def pooled_metrics_from_group(group):
    y_true = group["y_true"].to_numpy(dtype=float)
    y_pred = group["y_pred"].to_numpy(dtype=float)
    return {
        "pooled_outer_q2": safe_r2(y_true, y_pred),
        "pooled_outer_mae": mean_absolute_error(y_true, y_pred),
        "n_predictions": int(len(group)),
        "n_held_out_labels": int(group["held_out_label"].nunique()),
    }


pooled_rows = []
for group_keys, group in pred_df.groupby(["selection_mode", "descriptor_family", "validation_regime"]):
    pooled_rows.append({
        "selection_mode": group_keys[0],
        "descriptor_family": group_keys[1],
        "validation_regime": group_keys[2],
        **pooled_metrics_from_group(group),
    })

outer_pooled_stats = pd.DataFrame(pooled_rows)
outer_leave_one_class_q2 = outer_pooled_stats[
    outer_pooled_stats["validation_regime"] == "leave_one_type_out"
].reset_index(drop=True)


# -----------------------------
# 5) Fixed-spec pooling across all validation regimes
# -----------------------------
fixed_spec_pool_records = []
if not fixed_spec_df.empty and not fixed_spec_pred_df.empty:
    grouped_fixed_specs = fixed_spec_pred_df.groupby(["selection_mode", "descriptor_family", "validation_regime", "spec_key"])
    for group_keys, pred_group in grouped_fixed_specs:
        spec_meta_rows = fixed_spec_df[
            (fixed_spec_df["selection_mode"] == group_keys[0])
            & (fixed_spec_df["descriptor_family"] == group_keys[1])
            & (fixed_spec_df["validation_regime"] == group_keys[2])
            & (fixed_spec_df["spec_key"] == group_keys[3])
        ]
        pooled_metrics = pooled_metrics_from_group(pred_group)
        expected_label_count = len(validation_labels_by_regime.get(group_keys[2], []))
        fixed_spec_pool_records.append({
            "selection_mode": group_keys[0],
            "descriptor_family": group_keys[1],
            "validation_regime": group_keys[2],
            "spec_key": group_keys[3],
            "model": spec_meta_rows["model"].iloc[0],
            "k_selected": int(spec_meta_rows["k_selected"].iloc[0]),
            "selected_descriptors": spec_meta_rows["selected_descriptors"].iloc[0],
            "mean_rank_in_fold": float(spec_meta_rows["rank_in_fold"].mean()),
            **pooled_metrics,
            "full_regime_coverage": int(pred_group["held_out_label"].nunique()) == expected_label_count,
        })

fixed_spec_pool_results = pd.DataFrame(fixed_spec_pool_records)
if fixed_spec_pool_results.empty:
    fixed_spec_full_coverage = pd.DataFrame()
    best_fixed_spec_by_regime = pd.DataFrame()
else:
    fixed_spec_full_coverage = fixed_spec_pool_results[fixed_spec_pool_results["full_regime_coverage"]].copy()
    best_fixed_spec_by_regime = (
        fixed_spec_full_coverage
        .sort_values(["selection_mode", "descriptor_family", "validation_regime", "pooled_outer_q2", "pooled_outer_mae", "mean_rank_in_fold"], ascending=[True, True, True, False, True, True])
        .groupby(["selection_mode", "descriptor_family", "validation_regime"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )


# -----------------------------
# 6) Nested leave-one-type-out spec pooling
# -----------------------------
nested_leave_one_type_spec_pool_records = []
if not leave_one_type_spec_df.empty and not leave_one_type_spec_pred_df.empty:
    grouped_specs = leave_one_type_spec_pred_df.groupby(["selection_mode", "descriptor_family", "spec_key"])
    for group_keys, pred_group in grouped_specs:
        spec_meta_rows = leave_one_type_spec_df[
            (leave_one_type_spec_df["selection_mode"] == group_keys[0])
            & (leave_one_type_spec_df["descriptor_family"] == group_keys[1])
            & (leave_one_type_spec_df["spec_key"] == group_keys[2])
        ]
        pooled_metrics = pooled_metrics_from_group(pred_group)
        nested_leave_one_type_spec_pool_records.append({
            "selection_mode": group_keys[0],
            "descriptor_family": group_keys[1],
            "spec_key": group_keys[2],
            "model": spec_meta_rows["model"].iloc[0],
            "k_selected": int(spec_meta_rows["k_selected"].iloc[0]),
            "selected_descriptors": spec_meta_rows["selected_descriptors"].iloc[0],
            "mean_rank_in_fold": float(spec_meta_rows["rank_in_fold"].mean()),
            "max_rank_in_fold": int(spec_meta_rows["rank_in_fold"].max()),
            "mean_inner_cv_train_r2_mean": float(spec_meta_rows["inner_cv_train_r2_mean"].mean()),
            "mean_inner_cv_train_mae_mean": float(spec_meta_rows["inner_cv_train_mae_mean"].mean()),
            "mean_inner_cv_val_q2": float(spec_meta_rows["inner_cv_val_q2"].mean()),
            "mean_inner_cv_val_mae": float(spec_meta_rows["inner_cv_val_mae"].mean()),
            "inner_cv_splits": int(spec_meta_rows["inner_cv_splits"].iloc[0]),
            "inner_cv_repeats": int(spec_meta_rows["inner_cv_repeats"].iloc[0]),
            "inner_cv_evals": int(spec_meta_rows["inner_cv_evals"].iloc[0]),
            **pooled_metrics,
            "full_type_coverage": int(pred_group["held_out_label"].nunique()) == len(leave_one_type_labels),
        })

nested_leave_one_type_spec_pool_results = pd.DataFrame(nested_leave_one_type_spec_pool_records)
if nested_leave_one_type_spec_pool_results.empty:
    nested_leave_one_type_full_coverage = pd.DataFrame()
else:
    nested_leave_one_type_full_coverage = (
        nested_leave_one_type_spec_pool_results[nested_leave_one_type_spec_pool_results["full_type_coverage"]]
        .sort_values(["selection_mode", "descriptor_family", "pooled_outer_q2", "pooled_outer_mae"], ascending=[True, True, False, True])
        .reset_index(drop=True)
    )


# Aggregate statistics per descriptor family x validation regime
summary_stats = (
    results_df
    .groupby(["selection_mode", "descriptor_family", "validation_regime"], as_index=False)
    .agg(
        folds=("outer_test_mae", "size"),
        valid_outer_test_r2_folds=("outer_test_r2_stable", lambda s: int(s.notna().sum())),
        mean_outer_test_mae=("outer_test_mae", "mean"),
        std_outer_test_mae=("outer_test_mae", "std"),
        mean_inner_cv_val_q2=("inner_cv_val_q2", "mean"),
        mean_inner_cv_val_mae=("inner_cv_val_mae", "mean"),
    )
    .merge(
        best_fixed_spec_by_regime[["selection_mode", "descriptor_family", "validation_regime", "pooled_outer_q2", "spec_key", "selected_descriptors"]],
        on=["selection_mode", "descriptor_family", "validation_regime"],
        how="left",
    )
    .rename(columns={"pooled_outer_q2": "outer_test_q2", "spec_key": "q2_fixed_spec_key", "selected_descriptors": "q2_fixed_selected_descriptors"})
)

# Best fold-level model per descriptor family x validation regime by outer test MAE
best_models = (
    results_df
    .sort_values(["selection_mode", "descriptor_family", "validation_regime", "outer_test_mae", "outer_test_r2_stable"], ascending=[True, True, True, True, False])
    .groupby(["selection_mode", "descriptor_family", "validation_regime"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
summary_stats = summary_stats.merge(
    best_models[["selection_mode", "descriptor_family", "validation_regime", "model", "k_selected", "selected_descriptors"]].rename(
        columns={
            "model": "best_model",
            "k_selected": "best_k_selected",
            "selected_descriptors": "best_selected_descriptors",
        }
    ),
    on=["selection_mode", "descriptor_family", "validation_regime"],
    how="left",
)

print("Selection mode:", selection_mode)
print("Inner feature-selection split: y_equidistant with", inner_cv_n_splits, "splits")
print("Inner ranking metric: pooled out-of-fold Q2")
print("Max selected features:", max_selected_features)
print("DFT correlation filter threshold:", dft_correlation_threshold)
print("DFT descriptors before/after correlation filter:", dft_feature_count_before, "->", dft_feature_count_after)
print("DFT descriptors removed:", len(dft_removed_features))
print("Outer pooled leave-one-class metric: pooled Q2 across all held-out classes")
print("Summary-table outer_test_q2 metric: pooled Q2 from a single fixed descriptor specification per regime")
print("Nested spec-pool leave-one-type-out metric: pooled Q2 recombined only for matching no-leakage specs across folds")
print("Feature set shapes:")
for n, x in feature_sets.items():
    print(f"  {n}: {x.shape}")
print("\nCompleted model fits:", len(results_df))
print("Stored leave-one-type-out spec pool size per fold:", spec_pool_size)

# Refit a bank of top fixed descriptor specifications across every fold in each regime
# and redefine summary_stats outer_test_q2 from the best pooled-Q2 fixed spec.

if "fixed_spec_df" not in globals() or fixed_spec_df.empty:
    raise RuntimeError("Run Cell 5 first to create fixed_spec_df.")

fixed_spec_refit_pool_size = 10000

model_candidates_by_name = {name: model for name, model in get_model_candidates()}

spec_candidates_for_refit = (
    fixed_spec_df
    .groupby(["selection_mode", "descriptor_family", "validation_regime", "spec_key", "model", "k_selected"], as_index=False)
    .agg(
        selected_descriptors=("selected_descriptors", "first"),
        folds_seen=("held_out_label", "nunique"),
        mean_rank_in_fold=("rank_in_fold", "mean"),
        best_rank_in_fold=("rank_in_fold", "min"),
        mean_inner_cv_val_q2=("inner_cv_val_q2", "mean"),
        mean_inner_cv_val_mae=("inner_cv_val_mae", "mean"),
    )
    .sort_values(
        [
            "selection_mode",
            "descriptor_family",
            "validation_regime",
            "folds_seen",
            "mean_rank_in_fold",
            "best_rank_in_fold",
            "mean_inner_cv_val_q2",
            "mean_inner_cv_val_mae",
        ],
        ascending=[True, True, True, False, True, True, False, True],
    )
    .groupby(["selection_mode", "descriptor_family", "validation_regime"], as_index=False)
    .head(fixed_spec_refit_pool_size)
    .reset_index(drop=True)
 )


def refit_fixed_spec_across_regime(selection_mode, descriptor_family, validation_regime, model_name, selected_descriptors, spec_key):
    X_df = feature_sets[descriptor_family]
    descriptor_list = list(selected_descriptors)
    if any(desc not in X_df.columns for desc in descriptor_list):
        return None

    base_model = model_candidates_by_name.get(model_name)
    if base_model is None:
        return None

    prediction_rows = []
    for tr_idx, te_idx, held_out_label in outer_splits[validation_regime]:
        X_train = X_df.iloc[tr_idx][descriptor_list].to_numpy(dtype=float)
        X_test = X_df.iloc[te_idx][descriptor_list].to_numpy(dtype=float)
        y_train = y[tr_idx]
        y_test = y[te_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        fit_model = clone(base_model)
        fit_model.fit(X_train_scaled, y_train)
        y_pred = fit_model.predict(X_test_scaled)

        for sample_index, y_true_i, y_pred_i in zip(te_idx.tolist(), y_test.tolist(), y_pred.tolist()):
            prediction_rows.append({
                "held_out_label": held_out_label,
                "sample_index": int(sample_index),
                "y_true": float(y_true_i),
                "y_pred": float(y_pred_i),
            })

    if not prediction_rows:
        return None

    pred_group = pd.DataFrame(prediction_rows)
    pooled_metrics = pooled_metrics_from_group(pred_group)
    expected_label_count = len(validation_labels_by_regime.get(validation_regime, []))

    return {
        "selection_mode": selection_mode,
        "descriptor_family": descriptor_family,
        "validation_regime": validation_regime,
        "spec_key": spec_key,
        "model": model_name,
        "selected_descriptors": descriptor_list,
        **pooled_metrics,
        "full_regime_coverage": int(pred_group["held_out_label"].nunique()) == expected_label_count,
    }


refit_fixed_spec_pool_records = []
for _, spec_row in spec_candidates_for_refit.iterrows():
    refit_result = refit_fixed_spec_across_regime(
        selection_mode=spec_row["selection_mode"],
        descriptor_family=spec_row["descriptor_family"],
        validation_regime=spec_row["validation_regime"],
        model_name=spec_row["model"],
        selected_descriptors=spec_row["selected_descriptors"],
        spec_key=spec_row["spec_key"],
    )
    if refit_result is not None:
        refit_fixed_spec_pool_records.append(refit_result)

refit_fixed_spec_pool_results = pd.DataFrame(refit_fixed_spec_pool_records)
if refit_fixed_spec_pool_results.empty:
    best_refit_fixed_spec_by_regime = pd.DataFrame()
else:
    best_refit_fixed_spec_by_regime = (
        refit_fixed_spec_pool_results
        .sort_values(
            ["selection_mode", "descriptor_family", "validation_regime", "pooled_outer_q2", "pooled_outer_mae"],
            ascending=[True, True, True, False, True],
        )
        .groupby(["selection_mode", "descriptor_family", "validation_regime"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

summary_stats = (
    summary_stats
    .drop(columns=["outer_test_q2", "q2_fixed_spec_key", "q2_fixed_selected_descriptors"], errors="ignore")
    .merge(
        best_refit_fixed_spec_by_regime[[
            "selection_mode",
            "descriptor_family",
            "validation_regime",
            "pooled_outer_q2",
            "spec_key",
            "selected_descriptors",
        ]].rename(
            columns={
                "pooled_outer_q2": "outer_test_q2",
                "spec_key": "q2_fixed_spec_key",
                "selected_descriptors": "q2_fixed_selected_descriptors",
            }
        ),
        on=["selection_mode", "descriptor_family", "validation_regime"],
        how="left",
    )
 )

print("Refit fixed-spec candidate bank per regime:", fixed_spec_refit_pool_size)
print("outer_test_q2 now comes from refitting each banked spec across all folds and choosing the best pooled-Q2 spec.")


summary_stats

In [ ]:
# Re-rank fixed-spec candidates by selection_mode objective (not pooled outer Q2),
# while still reporting recomputed outer_test_q2 from refit predictions.

if "fixed_spec_df" not in globals() or fixed_spec_df.empty:
    raise RuntimeError("Run Cell 5 first to create fixed_spec_df.")
if "refit_fixed_spec_pool_results" not in globals() or refit_fixed_spec_pool_results.empty:
    raise RuntimeError("Run Cell 5 first to create refit_fixed_spec_pool_results.")

# Rebuild candidate metadata with inner-CV aggregate metrics used for selection ranking.
spec_candidates_for_selection = (
    fixed_spec_df
    .groupby(["selection_mode", "descriptor_family", "validation_regime", "spec_key", "model", "k_selected"], as_index=False)
    .agg(
        selected_descriptors=("selected_descriptors", "first"),
        folds_seen=("held_out_label", "nunique"),
        mean_rank_in_fold=("rank_in_fold", "mean"),
        best_rank_in_fold=("rank_in_fold", "min"),
        mean_inner_cv_train_r2_mean=("inner_cv_train_r2_mean", "mean"),
        mean_inner_cv_train_mae_mean=("inner_cv_train_mae_mean", "mean"),
        mean_inner_cv_val_q2=("inner_cv_val_q2", "mean"),
        mean_inner_cv_val_mae=("inner_cv_val_mae", "mean"),
    )
    .reset_index(drop=True)
)

spec_candidates_for_selection["selection_score"] = spec_candidates_for_selection.apply(
    lambda row: rank_candidate(
        train_r2=row["mean_inner_cv_train_r2_mean"],
        train_mae=row["mean_inner_cv_train_mae_mean"],
        val_q2=row["mean_inner_cv_val_q2"],
        val_mae=row["mean_inner_cv_val_mae"],
        mode=row["selection_mode"],
    ),
    axis=1,
)

# Attach selection-mode ranking metadata to refit results.
refit_fixed_spec_pool_results = refit_fixed_spec_pool_results.drop(
    columns=[
        "selection_score",
        "folds_seen",
        "mean_rank_in_fold",
        "best_rank_in_fold",
        "mean_inner_cv_train_r2_mean",
        "mean_inner_cv_train_mae_mean",
        "mean_inner_cv_val_q2",
        "mean_inner_cv_val_mae",
    ],
    errors="ignore",
).merge(
    spec_candidates_for_selection[[
        "selection_mode",
        "descriptor_family",
        "validation_regime",
        "spec_key",
        "selection_score",
        "folds_seen",
        "mean_rank_in_fold",
        "best_rank_in_fold",
        "mean_inner_cv_train_r2_mean",
        "mean_inner_cv_train_mae_mean",
        "mean_inner_cv_val_q2",
        "mean_inner_cv_val_mae",
    ]],
    on=["selection_mode", "descriptor_family", "validation_regime", "spec_key"],
    how="left",
)

# Select winning fixed spec by selection objective, not outer_test_q2.
best_refit_fixed_spec_by_regime = (
    refit_fixed_spec_pool_results
    .sort_values(
        [
            "selection_mode",
            "descriptor_family",
            "validation_regime",
            "selection_score",
            "folds_seen",
            "mean_rank_in_fold",
            "best_rank_in_fold",
            "mean_inner_cv_val_q2",
            "mean_inner_cv_val_mae",
        ],
        ascending=[True, True, True, False, False, True, True, False, True],
    )
    .groupby(["selection_mode", "descriptor_family", "validation_regime"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# Keep outer_test_q2 as recomputed pooled refit Q2 for the selected fixed spec.
summary_stats = (
    summary_stats
    .drop(columns=["outer_test_q2", "q2_fixed_spec_key", "q2_fixed_selected_descriptors"], errors="ignore")
    .merge(
        best_refit_fixed_spec_by_regime[[
            "selection_mode",
            "descriptor_family",
            "validation_regime",
            "pooled_outer_q2",
            "spec_key",
            "selected_descriptors",
            "selection_score",
        ]].rename(
            columns={
                "pooled_outer_q2": "outer_test_q2",
                "spec_key": "q2_fixed_spec_key",
                "selected_descriptors": "q2_fixed_selected_descriptors",
            }
        ),
        on=["selection_mode", "descriptor_family", "validation_regime"],
        how="left",
    )
)

print("Best fixed specs now selected by selection_mode objective; outer_test_q2 remains pooled refit Q2 for those selected specs.")
summary_stats

In [ ]:
# Detailed best-model report (descriptors + coefficients)
report_cols = [
    "selection_mode",
    "inner_cv_splits",
    "inner_cv_repeats",
    "inner_cv_evals",
    "descriptor_family",
    "validation_regime",
    "held_out_label",
    "model",
    "k_selected",
    "selected_descriptors",
    "coefficients",
    "intercept",
    "inner_cv_train_r2_mean",
    "inner_cv_train_mae_mean",
    "inner_cv_val_q2",
    "inner_cv_val_mae",
    "outer_test_r2",
    "outer_test_r2_stable",
    "outer_test_mae",
]

best_models[report_cols].sort_values(["selection_mode", "descriptor_family", "validation_regime"]).reset_index(drop=True)

In [ ]:
from IPython.display import display
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.rdFingerprintGenerator import AdditionalOutput

if "summary_stats" not in globals():
    raise RuntimeError("Run the main pipeline cell first to create summary_stats.")

if "work" not in globals():
    raise RuntimeError("Run the main pipeline cell first to create the aligned dataset 'work'.")

morgan_summary = summary_stats[summary_stats["descriptor_family"] == "Morgan"].copy()
if morgan_summary.empty:
    raise RuntimeError("No Morgan rows found in summary_stats.")

morgan_summary = morgan_summary.sort_values(["outer_test_q2", "mean_outer_test_mae"], ascending=[False, True]).reset_index(drop=True)


def parse_morgan_bits(descriptor_values):
    bits = []
    if isinstance(descriptor_values, (list, tuple, np.ndarray, pd.Series)):
        for desc in descriptor_values:
            desc_str = str(desc)
            if desc_str.startswith("morgan_"):
                try:
                    bits.append(int(desc_str.split("_", 1)[1]))
                except ValueError:
                    continue
    return bits


def draw_first_example_for_bit(bit_id, smiles_values):
    for smi in smiles_values:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue

        ao = AdditionalOutput()
        ao.CollectBitInfoMap()
        _ = morgan_generator.GetFingerprint(mol, additionalOutput=ao)
        bit_info = ao.GetBitInfoMap()

        if bit_id in bit_info:
            img = Draw.DrawMorganBit(mol, bit_id, bit_info, whichExample=0, useSVG=False)
            return img, smi
    return None, None


smiles_values = work["computational_smiles"].astype(str).tolist()

print("Morgan bit drawings for best selected features in summary_stats")
print("Rows are sorted by highest outer_test_q2 then lowest mean_outer_test_mae.\n")

for _, row in morgan_summary.iterrows():
    selected_bits = parse_morgan_bits(row.get("best_selected_descriptors", []))
    print(
        f"Validation regime: {row['validation_regime']} | outer_test_q2={row['outer_test_q2']:.3f} | "
        f"best_k_selected={row['best_k_selected']}"
    )

    if not selected_bits:
        print("  No Morgan bit descriptors found in best_selected_descriptors.\n")
        continue

    for bit_id in selected_bits:
        img, matched_smi = draw_first_example_for_bit(bit_id, smiles_values)
        if img is None:
            print(f"  morgan_{bit_id}: no matching molecule found in current dataset")
            continue
        print(f"  morgan_{bit_id} | example SMILES: {matched_smi}")
        display(img)
    print()

morgan_summary

In [ ]:
# Parity plots: top-3 fixed specs by balanced selection score per (validation regime, descriptor family)
# Legend shows balanced score, Q2_validation (inner), and Q2_test (pooled held-out parity).

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

required_vars = ["outer_splits", "feature_sets", "meta", "y"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Run Cell 5 first. Missing required variables: {missing}")


def safe_r2_local(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.isclose(np.std(y_true), 0.0):
        return np.nan
    return r2_score(y_true, y_pred)


def rank_candidate_local(train_r2, train_mae, val_q2, val_mae, mode):
    if "rank_candidate" in globals():
        return rank_candidate(train_r2, train_mae, val_q2, val_mae, mode)
    if mode == "r2":
        return -1.0 if np.isnan(val_q2) else val_q2
    if mode == "mae":
        return -val_mae
    if mode == "balanced":
        v_q2 = -1.0 if np.isnan(val_q2) else val_q2
        t_r2 = -1.0 if np.isnan(train_r2) else train_r2
        return (
            v_q2
            - val_mae
            - 0.5 * abs(t_r2 - v_q2)
            - 0.5 * abs(train_mae - val_mae)
        )
    raise ValueError("selection_mode must be one of: 'r2', 'mae', 'balanced'")


# Build ranking table with one row per fixed spec and balanced selection metadata.
if "refit_fixed_spec_pool_results" in globals() and not refit_fixed_spec_pool_results.empty:
    spec_table = refit_fixed_spec_pool_results.copy()
else:
    if "fixed_spec_df" not in globals() or fixed_spec_df.empty:
        raise RuntimeError("Need either refit_fixed_spec_pool_results or fixed_spec_df. Run Cell 5 first.")

    spec_table = (
        fixed_spec_df
        .groupby(["selection_mode", "descriptor_family", "validation_regime", "spec_key", "model", "k_selected"], as_index=False)
        .agg(
            selected_descriptors=("selected_descriptors", "first"),
            mean_inner_cv_train_r2_mean=("inner_cv_train_r2_mean", "mean"),
            mean_inner_cv_train_mae_mean=("inner_cv_train_mae_mean", "mean"),
            mean_inner_cv_val_q2=("inner_cv_val_q2", "mean"),
            mean_inner_cv_val_mae=("inner_cv_val_mae", "mean"),
            mean_rank_in_fold=("rank_in_fold", "mean"),
            folds_seen=("held_out_label", "nunique"),
        )
        .reset_index(drop=True)
    )

# Ensure ranking metrics exist for balanced-score sorting.
if "mean_inner_cv_train_r2_mean" not in spec_table.columns or "mean_inner_cv_train_mae_mean" not in spec_table.columns:
    if "fixed_spec_df" in globals() and not fixed_spec_df.empty:
        train_metrics = (
            fixed_spec_df
            .groupby(["selection_mode", "descriptor_family", "validation_regime", "spec_key"], as_index=False)
            .agg(
                mean_inner_cv_train_r2_mean=("inner_cv_train_r2_mean", "mean"),
                mean_inner_cv_train_mae_mean=("inner_cv_train_mae_mean", "mean"),
            )
        )
        spec_table = spec_table.merge(
            train_metrics,
            on=["selection_mode", "descriptor_family", "validation_regime", "spec_key"],
            how="left",
        )
    else:
        spec_table["mean_inner_cv_train_r2_mean"] = np.nan
        spec_table["mean_inner_cv_train_mae_mean"] = np.nan

if "mean_inner_cv_val_q2" not in spec_table.columns or "mean_inner_cv_val_mae" not in spec_table.columns:
    if "fixed_spec_df" in globals() and not fixed_spec_df.empty:
        inner_metrics = (
            fixed_spec_df
            .groupby(["selection_mode", "descriptor_family", "validation_regime", "spec_key"], as_index=False)
            .agg(
                mean_inner_cv_val_q2=("inner_cv_val_q2", "mean"),
                mean_inner_cv_val_mae=("inner_cv_val_mae", "mean"),
            )
        )
        spec_table = spec_table.merge(
            inner_metrics,
            on=["selection_mode", "descriptor_family", "validation_regime", "spec_key"],
            how="left",
        )
    else:
        raise RuntimeError("Cannot compute balanced-score ranking. fixed_spec_df is missing.")

if "mean_rank_in_fold" not in spec_table.columns:
    spec_table["mean_rank_in_fold"] = np.nan
if "folds_seen" not in spec_table.columns:
    if "fixed_spec_df" in globals() and not fixed_spec_df.empty:
        fold_counts = (
            fixed_spec_df
            .groupby(["selection_mode", "descriptor_family", "validation_regime", "spec_key"], as_index=False)
            .agg(folds_seen=("held_out_label", "nunique"))
        )
        spec_table = spec_table.merge(
            fold_counts,
            on=["selection_mode", "descriptor_family", "validation_regime", "spec_key"],
            how="left",
        )
    else:
        spec_table["folds_seen"] = np.nan

if "selection_score" not in spec_table.columns:
    spec_table["selection_score"] = spec_table.apply(
        lambda row: rank_candidate_local(
            train_r2=row["mean_inner_cv_train_r2_mean"],
            train_mae=row["mean_inner_cv_train_mae_mean"],
            val_q2=row["mean_inner_cv_val_q2"],
            val_mae=row["mean_inner_cv_val_mae"],
            mode=row["selection_mode"],
        ),
        axis=1,
    )


def rank_top_specs_for_panel(pool_df, top_k=3):
    ranked = (
        pool_df
        .sort_values(
            [
                "selection_mode",
                "descriptor_family",
                "validation_regime",
                "selection_score",
                "folds_seen",
                "mean_inner_cv_val_q2",
                "mean_inner_cv_val_mae",
                "mean_rank_in_fold",
            ],
            ascending=[True, True, True, False, False, False, True, True],
        )
        .groupby(["selection_mode", "descriptor_family", "validation_regime"], as_index=False)
        .head(top_k)
        .copy()
    )
    ranked["rank_in_panel"] = (
        ranked.groupby(["selection_mode", "descriptor_family", "validation_regime"]).cumcount() + 1
    )
    return ranked


top_k_specs = 3
ranked_top_specs = rank_top_specs_for_panel(spec_table, top_k=top_k_specs)

selection_mode_for_plot = None
if "selection_mode" in summary_stats.columns and not summary_stats.empty:
    selection_mode_for_plot = summary_stats["selection_mode"].mode().iloc[0]

if selection_mode_for_plot is not None:
    ranked_top_specs = ranked_top_specs[ranked_top_specs["selection_mode"] == selection_mode_for_plot].copy()

if ranked_top_specs.empty:
    raise RuntimeError("No ranked top specs available after filtering. Check selection_mode alignment.")

regime_order = ["random", "y_equidistant", "leave_one_ligandID_out", "leave_one_type_out", "leave_one_substituent_out"]
regimes_present = [r for r in regime_order if r in ranked_top_specs["validation_regime"].unique().tolist()]
descriptor_order = [d for d in ["DFT", "Morgan", "TypeOHE"] if d in ranked_top_specs["descriptor_family"].unique().tolist()]

all_types_global = sorted(meta["type"].astype(str).unique().tolist())
marker_cycle = ["o", "s", "^", "D", "v", "P", "X", "*"]
type_to_marker = {t: marker_cycle[i % len(marker_cycle)] for i, t in enumerate(all_types_global)}

model_candidates_by_name = {name: model for name, model in get_model_candidates()}

for regime in regimes_present:
    fig, axes = plt.subplots(1, len(descriptor_order), figsize=(6 * len(descriptor_order), 6), sharex=True, sharey=True)
    if len(descriptor_order) == 1:
        axes = [axes]

    fig.suptitle(
        f"Top-{top_k_specs} balanced-selection specs | test_regime={regime} | color=rank, marker=ligand class",
        y=0.99,
    )

    for ax, fam in zip(axes, descriptor_order):
        panel_specs = ranked_top_specs[
            (ranked_top_specs["validation_regime"] == regime) &
            (ranked_top_specs["descriptor_family"] == fam)
        ].copy()

        if panel_specs.empty:
            ax.set_title(f"{fam} (no top specs)")
            ax.axis("off")
            continue

        panel_specs = panel_specs.sort_values("rank_in_panel").reset_index(drop=True)
        n_specs = len(panel_specs)
        set_colors = plt.cm.tab10(np.linspace(0, 1, max(n_specs, 3)))

        axis_values = []
        score_by_rank = {}
        q2_test_by_rank = {}
        q2_val_by_rank = {}

        print(f"\n{fam} | {regime}:")

        for _, spec_row in panel_specs.iterrows():
            rank_idx = int(spec_row["rank_in_panel"])
            desc_list = list(spec_row["selected_descriptors"])
            model_name = spec_row["model"]
            spec_key = spec_row["spec_key"]
            selection_score = float(spec_row["selection_score"]) if pd.notna(spec_row["selection_score"]) else np.nan
            q2_val = float(spec_row["mean_inner_cv_val_q2"]) if pd.notna(spec_row["mean_inner_cv_val_q2"]) else np.nan
            score_by_rank[rank_idx] = selection_score
            q2_val_by_rank[rank_idx] = q2_val

            X_df = feature_sets[fam]
            if any(d not in X_df.columns for d in desc_list):
                print(f"  rank {rank_idx} skipped (missing descriptor columns): {spec_key}")
                continue

            base_model = model_candidates_by_name.get(model_name)
            if base_model is None:
                print(f"  rank {rank_idx} skipped (model not found): {model_name}")
                continue

            all_records_set = []
            for tr_idx, te_idx, _ in outer_splits[regime]:
                X_train_set = X_df.iloc[tr_idx][desc_list].to_numpy(dtype=float)
                X_test_set = X_df.iloc[te_idx][desc_list].to_numpy(dtype=float)
                y_train_data = y[tr_idx]
                y_test_data = y[te_idx]

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_set)
                X_test_scaled = scaler.transform(X_test_set)

                model = clone(base_model)
                model.fit(X_train_scaled, y_train_data)
                y_pred_test = model.predict(X_test_scaled)

                te_types = meta.iloc[te_idx]["type"].astype(str).tolist()
                for y_true_i, y_pred_i, lig_type_i in zip(y_test_data.tolist(), y_pred_test.tolist(), te_types):
                    all_records_set.append(
                        {
                            "y_true": float(y_true_i),
                            "y_pred": float(y_pred_i),
                            "ligand_type": lig_type_i,
                        }
                    )

            if not all_records_set:
                continue

            set_plot_df = pd.DataFrame(all_records_set)
            y_true_set = set_plot_df["y_true"].to_numpy(dtype=float)
            y_pred_set = set_plot_df["y_pred"].to_numpy(dtype=float)

            q2_test = safe_r2_local(y_true_set, y_pred_set)
            mae_set = mean_absolute_error(y_true_set, y_pred_set)
            n_set = len(y_true_set)
            q2_test_by_rank[rank_idx] = q2_test
            axis_values.extend(y_true_set.tolist())
            axis_values.extend(y_pred_set.tolist())

            desc_names = ", ".join(desc_list[:4]) + ("..." if len(desc_list) > 4 else "")
            print(
                f"  rank {rank_idx} | model={model_name} | {desc_names} | n={n_set}: "
                f"score={selection_score:.3f}, Q2_validation={q2_val:.3f}, Q2_test={q2_test:.3f}, MAE={mae_set:.3f}"
            )

            color_i = set_colors[rank_idx - 1]
            for lig_type in sorted(set_plot_df["ligand_type"].unique()):
                sub_t = set_plot_df[set_plot_df["ligand_type"] == lig_type]
                ax.scatter(
                    sub_t["y_true"].to_numpy(dtype=float),
                    sub_t["y_pred"].to_numpy(dtype=float),
                    alpha=0.75,
                    s=52,
                    c=[color_i],
                    marker=type_to_marker.get(lig_type, "o"),
                    edgecolors="none",
                )

        if axis_values:
            lim_min = float(np.min(axis_values))
            lim_max = float(np.max(axis_values))
        else:
            lim_min, lim_max = -1.0, 1.0

        pad = 0.5 if np.isclose(lim_min, lim_max) else 0.05 * (lim_max - lim_min)
        lim_min -= pad
        lim_max += pad

        ax.plot([lim_min, lim_max], [lim_min, lim_max], "k--", lw=1, alpha=0.5)
        ax.set_xlim(lim_min, lim_max)
        ax.set_ylim(lim_min, lim_max)
        ax.set_aspect("equal", adjustable="box")
        if hasattr(ax, "set_box_aspect"):
            ax.set_box_aspect(1)

        ax.set_xlabel("True ddg_flipped")
        ax.set_ylabel("Predicted ddg_flipped")
        ax.grid(True, alpha=0.3)
        ax.set_title(f"{fam}\n(top {n_specs} by balanced selection)")

        color_handles = []
        for i in range(1, n_specs + 1):
            score_i = score_by_rank.get(i, np.nan)
            q2v_i = q2_val_by_rank.get(i, np.nan)
            q2t_i = q2_test_by_rank.get(i, np.nan)
            if pd.notna(score_i) and pd.notna(q2v_i) and pd.notna(q2t_i):
                label = f"rank {i} (score={score_i:.3f}, Q2_validation={q2v_i:.3f}, Q2_test={q2t_i:.3f})"
            else:
                label = f"rank {i}"
            color_handles.append(
                Line2D(
                    [0],
                    [0],
                    marker="o",
                    linestyle="None",
                    color=set_colors[i - 1],
                    markersize=6,
                    label=label,
                )
            )

        marker_handles = [
            Line2D(
                [0],
                [0],
                marker=type_to_marker[t],
                color="black",
                linestyle="None",
                markersize=6,
                label=t,
            )
            for t in all_types_global
        ]

        if color_handles:
            color_legend = ax.legend(
                handles=color_handles,
                title="Top specs by balanced selection",
                loc="lower right",
                frameon=True,
                fontsize=8,
                title_fontsize=8,
            )
            ax.add_artist(color_legend)

        if marker_handles:
            ax.legend(
                handles=marker_handles,
                title="Ligand class (marker)",
                loc="upper left",
                frameon=True,
                fontsize=8,
                title_fontsize=8,
            )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()